# BuySense - Random Forest Modelling

This notebook develops and evaluates a Random Forest classifier for predicting online purchase intention.

The objectives are to:

- build a baseline Random Forest model
- evaluate the model using 5-fold stratified cross-validation
- compare normal and balanced class weights
- tune important Random Forest hyperparameters using RandomizedSearchCV
- compare baseline and tuned model performance
- investigate overfitting using training and validation scores
- examine Random Forest feature importance
- compare model performance with and without PageValues
- document findings for final model comparison

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    make_scorer
)

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/raw/online_shoppers_intention.csv")

print(df.shape)
df.head()

(12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [3]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [5]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain purchase rate:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest purchase rate:")
print(y_test.value_counts(normalize=True) * 100)

Train shape: (9864, 17)
Test shape: (2466, 17)

Train purchase rate:
Revenue
0    84.529603
1    15.470397
Name: proportion, dtype: float64

Test purchase rate:
Revenue
0    84.509327
1    15.490673
Name: proportion, dtype: float64


In [6]:
def add_engineered_features(data):
    data = data.copy()

    data["TotalPages"] = (
        data["Administrative"]
        + data["Informational"]
        + data["ProductRelated"]
    )

    data["TotalDuration"] = (
        data["Administrative_Duration"]
        + data["Informational_Duration"]
        + data["ProductRelated_Duration"]
    )

    data["AvgDurationPerPage"] = np.where(
        data["TotalPages"] > 0,
        data["TotalDuration"] / data["TotalPages"],
        0
    )

    data["ProductPageShare"] = np.where(
        data["TotalPages"] > 0,
        data["ProductRelated"] / data["TotalPages"],
        0
    )

    data["ProductDurationShare"] = np.where(
        data["TotalDuration"] > 0,
        data["ProductRelated_Duration"] / data["TotalDuration"],
        0
    )

    return data

In [7]:
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

In [8]:
engineered_features = [
    "TotalPages",
    "TotalDuration",
    "AvgDurationPerPage",
    "ProductPageShare",
    "ProductDurationShare"
]

print("Missing values:")
print(X_train[engineered_features].isna().sum())

print("\nInfinite values:")
print(np.isinf(X_train[engineered_features]).sum())

Missing values:
TotalPages              0
TotalDuration           0
AvgDurationPerPage      0
ProductPageShare        0
ProductDurationShare    0
dtype: int64

Infinite values:
TotalPages              0
TotalDuration           0
AvgDurationPerPage      0
ProductPageShare        0
ProductDurationShare    0
dtype: int64


In [9]:
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

boolean_features = ["Weekend"]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + boolean_features
]

In [10]:
print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nBoolean features:")
print(boolean_features)

Numerical features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'TotalPages', 'TotalDuration', 'AvgDurationPerPage', 'ProductPageShare', 'ProductDurationShare']

Categorical features:
['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']

Boolean features:
['Weekend']


In [11]:
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        ("bool", "passthrough", boolean_features)
    ]
)

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [14]:
def summarize_cv(model_name, pipeline, X_data, y_data):

    scores = cross_validate(
        pipeline,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    row = {
        "Model": model_name
    }

    for metric in scoring:
        row[f"CV_{metric}_mean"] = scores[
            f"test_{metric}"
        ].mean()

        row[f"CV_{metric}_std"] = scores[
            f"test_{metric}"
        ].std()

    row["Train_F1_mean"] = scores[
        "train_f1"
    ].mean()

    row["Validation_F1_mean"] = scores[
        "test_f1"
    ].mean()

    return row, scores

In [15]:
rf_baseline = Pipeline([
    ("preprocessor", preprocessor_tree),

    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

In [16]:
rf_baseline_row, rf_baseline_scores = summarize_cv(
    "Random Forest - Baseline",
    rf_baseline,
    X_train,
    y_train
)

In [17]:
pd.DataFrame([rf_baseline_row])

,Model,CV_accuracy_mean,CV_accuracy_std,CV_precision_mean,CV_precision_std,CV_recall_mean,CV_recall_std,CV_f1_mean,CV_f1_std,CV_roc_auc_mean,CV_roc_auc_std,CV_pr_auc_mean,CV_pr_auc_std,Train_F1_mean,Validation_F1_mean
0,Random Forest - Baseline,0.900751,0.006302,0.769004,0.025051,0.511786,0.028127,0.614457,0.027821,0.923473,0.005264,0.735803,0.019361,1.0,0.614457


In [18]:
pd.DataFrame([rf_baseline_row])[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean",
        "Train_F1_mean",
        "Validation_F1_mean"
    ]
]

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean,Train_F1_mean,Validation_F1_mean
0,Random Forest - Baseline,0.900751,0.769004,0.511786,0.614457,0.923473,0.735803,1.0,0.614457


### Baseline Random Forest Observation

The baseline Random Forest achieved **90.08% mean cross-validation accuracy** and a strong **ROC-AUC of 0.923**, showing good overall ability to separate purchasers from non-purchasers.

The model achieved **precision of 0.769**, meaning most sessions predicted as purchases were correct. However, **recall was lower at 0.512**, so the model identified only about half of the actual purchasers.

The **F1-score was 0.614** and the **PR-AUC was 0.736**, indicating reasonable performance on the minority purchase class.

However, the **training F1-score was 1.00**, while the **validation F1-score was 0.614**. This large gap suggests that the baseline Random Forest is **overfitting** the training data and may need tuning to improve generalization.

## Random Forest with Balanced Class Weights

The baseline Random Forest showed lower recall for the minority purchase class. Therefore, a second Random Forest model is tested using `class_weight="balanced"` to give greater importance to the minority purchase class.

In [19]:
rf_balanced = Pipeline([
    ("preprocessor", preprocessor_tree),

    ("model", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [20]:
rf_balanced_row, rf_balanced_scores = summarize_cv(
    "Random Forest - Balanced",
    rf_balanced,
    X_train,
    y_train
)

In [21]:
pd.DataFrame([rf_balanced_row])[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean",
        "Train_F1_mean",
        "Validation_F1_mean"
    ]
]

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean,Train_F1_mean,Validation_F1_mean
0,Random Forest - Balanced,0.894161,0.644982,0.710355,0.675401,0.925779,0.721527,0.997386,0.675401


In [22]:
rf_classweight_compare = pd.DataFrame([
    rf_baseline_row,
    rf_balanced_row
])

rf_classweight_compare[
    [
        "Model",
        "CV_accuracy_mean",
        "CV_precision_mean",
        "CV_recall_mean",
        "CV_f1_mean",
        "CV_roc_auc_mean",
        "CV_pr_auc_mean",
        "Train_F1_mean",
        "Validation_F1_mean"
    ]
]

,Model,CV_accuracy_mean,CV_precision_mean,CV_recall_mean,CV_f1_mean,CV_roc_auc_mean,CV_pr_auc_mean,Train_F1_mean,Validation_F1_mean
0,Random Forest - Baseline,0.900751,0.769004,0.511786,0.614457,0.923473,0.735803,1.000000,0.614457
1,Random Forest - Balanced,0.894161,0.644982,0.710355,0.675401,0.925779,0.721527,0.997386,0.675401


### Balanced Random Forest Observation

Using `class_weight="balanced"` improved the model’s ability to identify actual purchasers.

Compared with the baseline model, **recall increased from 0.512 to 0.710** and the **F1-score improved from 0.614 to 0.675**. This means the balanced model detected a much larger share of the minority Purchase class.

However, **precision decreased from 0.769 to 0.645** and accuracy fell slightly from **90.08% to 89.42%**, showing the expected precision-recall trade-off.

The **ROC-AUC remained strong and improved slightly from 0.923 to 0.926**, while **PR-AUC decreased slightly from 0.736 to 0.722**.

The training F1 remained very high at **0.997**, compared with a validation F1 of **0.675**, so some overfitting is still present.

Overall, balanced class weights improved minority-class recall and F1-score, but at the cost of lower precision and slightly lower PR-AUC.